In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os
from pathlib import Path

In [16]:
import kagglehub

path = kagglehub.dataset_download("kapitanov/hagrid", output_dir="/Data")
print("Dataset saved to:", path)


100%|██████████| 338M/338M [00:32<00:00, 10.8MB/s] 

Extracting files...


Dataset saved to: /Data


In [ ]:


# get all files in the Data directory

# files = Path("Data").glob("*")


# frames = []

# for file in sorted(Path("Data").glob("*.json")):
#     try:
#         frame = pd.read_json(file).T.rename_axis("id").reset_index()
#         frame["source_file"] = str(file)
#         frames.append(frame)
#     except (ValueError, OSError) as e:
#         print(f"Error reading {file}: {e}")

# collected_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# collected_df.to_csv("Data/collected_data.csv", index=False)


In [27]:
dx=pd.read_csv("Data/data.csv")

folders = ['train_val_call', 'train_val_dislike', 'train_val_fist', 'train_val_four', 'train_val_like']

List_of_files = []
files = []
labels = []
for folder in folders:
    folder_path = f"hagrid-sample-30k-384p/hagrid_30k/{folder}"
    files = sorted(os.listdir(folder_path))
    for file in files:
        folder_file_path = f"{folder_path}/{file}"
        List_of_files.append(folder_file_path)
        labels.append(folder)

mapping_frame= pd.DataFrame({"file_path": List_of_files})
mapping_frame["id"]= mapping_frame["file_path"].str.split("/").str[-1]
mapping_frame["id"]= mapping_frame["id"].str.split(".").str[0]
df= pd.merge(dx, mapping_frame, on="id", how="left")





X=pd.DataFrame(df["file_path"], columns=["file_path"])
y=pd.DataFrame(df["label"], columns=["label"])
label_to_id = {name: idx for idx, name in enumerate(sorted(y["label"].unique()))}


y["label"] = y["label"].map(label_to_id)


keys=pd.DataFrame(label_to_id.items(), columns=["label", "id"])

keys.to_csv("Data/label_to_id_mapping.csv", index=False)

In [ ]:

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)
Test_set=pd.DataFrame({"X": X_val["file_path"], "y": y_val["label"]})
Train_set=pd.DataFrame({"X": X_train["file_path"], "y": y_train["label"]})


Test_set.to_csv("Data/Test_set.csv", index=False)
Train_set.to_csv("Data/Train_set.csv", index=False)
